# Comparing Loss Functions for a Fixed CNN Reconstructor

`CNNArchitectureComparison.ipynb`, `AdvancedArchitectureComparison.ipynb`, `ActivationComparison.ipynb`, and `NormalizationComparison.ipynb` all hold the training *objective* fixed (`LogResidualVarianceLoss(pupil, wavelength) + Physics_loss(wfs=wfs)`) and vary the reconstructor instead. This notebook does the opposite: it holds the reconstructor fixed -- `ClassicCNN`, copied verbatim from `CNNArchitectureComparison.ipynb` (same `base_channels=28` default, same `num_downsamples`-derived depth) -- and sweeps only the loss function passed to `AI4AO.Trainer.Trainer`, training and evaluating each candidate following the exact same procedure as every notebook in this folder (same instrument, same `TrainRunNb`).

**The seven candidates**, spanning three different loss "domains":

- **Modal-coefficient regression** -- `ModalMAELoss` (L1) and `ModalMSELoss` (L2) directly supervise the reconstructor's predicted modal coefficients (`z_estimated`) against the ground-truth residual modal coefficients (`Ze`) -- the two most direct regression losses this pipeline can express.
- **Residual-phase regression** -- `ResidualPhaseMAELoss` and `ResidualPhaseMSELoss` are the phase-domain analogues of the two above: instead of comparing modal coefficients, they compare the post-correction residual phase (`corrected_residual_opd`, converted to radians via the WFS's own sensing wavelength) directly against zero, pixel-by-pixel over the pupil.
- **Physically-motivated losses** -- `LogResidualVarianceLoss` and `Physics_loss` are reused unmodified from `AI4AO/LossFunctions.py` (the loss every other notebook in this folder already trains with); `NegativeStrehlLoss` is new (defined below): the same post-correction residual-phase variance `LogResidualVarianceLoss` uses, but passed through the Marechal approximation for Strehl ratio, `S ~= exp(-var(phase))`, instead of a log. Minimizing `-exp(-var(phase))` directly maximizes this Strehl approximation, and -- unlike the log-variance loss, whose magnitude keeps growing without bound as the variance shrinks -- saturates near `-1` as the correction improves, mirroring how Strehl itself saturates near `1`.

All seven subclass the same `AOLoss` base every other loss in `AI4AO/LossFunctions.py` already uses, sharing its `(Ze, z_estimated, pupil, residual_opd, corrected_residual_opd, wfs_frames)` `forward` signature -- this is what lets any one of them drop straight into `Trainer`'s `loss` argument unchanged, exactly like `LogResidualVarianceLoss`/`Physics_loss` already do in the sibling notebooks.

**A comparison-methodology note, specific to this notebook**: every sibling notebook in this folder holds the loss function fixed and only varies the *network*, so their multi-way training-loss overlays are all measuring the same quantity and can be shown on one shared axis. Here the opposite is true -- each candidate loss is a genuinely different physical quantity in different units (radians, mode coefficients, log-variance, a bounded Strehl-like ratio), so overlaying their raw training curves on one axis would be meaningless. Training curves below are instead shown one panel per loss, each against its own oracle bound (computed by that same loss instance, so the two curves in a panel are directly comparable to each other). The one truly common, physically meaningful comparison across all seven is the seeded closed-loop rollout at the end: steady-state residual wavefront error (nm RMS) and the actual post-training Strehl approximation, both computed identically regardless of which loss trained the network.

This notebook reuses `CNNComparison_params.py` directly (same fictional-demo, nominal-geometry modulated Pyramid WFS + DM as every other notebook in this folder), so every comparison here runs on the identical instrument.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

from AI4AO.paths import DATA_DIR
from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import AOLoss, LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Configuration and instrument

Reuses `CNNComparison_params.py` unchanged -- the same single fictional-demo, nominal-geometry modulated Pyramid WFS + DM (no real bench) as every sibling notebook in `Tutorials/Advanced/`. `wfs` and `dm` are built fresh and frozen (`.eval()`); only each loss variant's `ClassicCNN` weights are ever trained. Checkpoints are kept under their own `Data/LossFunctionComparison/` directory, since the seven models trained here are a different set of weights from any sibling notebook's.

In [ ]:
paramfile = 'CNNComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = DATA_DIR / "LossFunctionComparison"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")

## `ClassicCNN` -- held fixed

Copied verbatim from `CNNArchitectureComparison.ipynb`: a plain, non-grouped `Conv2d` stack (VGG-style stages) directly on the 4 registered pupil channels, its depth picked from the pupil crop's actual resolution via `num_downsamples`, `base_channels=28` default (~0.9M trainable parameters). Every loss variant below trains its own fresh instance of this exact same architecture -- capacity and structure never change in this notebook, so any difference in the final comparison isolates the training objective alone.

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can
    take before dropping below min_size pixels -- same helper as
    CNNArchitectureComparison.ipynb's ClassicCNN, reused unmodified here."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def conv_stage(in_channels, out_channels, activation=nn.LeakyReLU):
    """Two 3x3 convs at out_channels, then a 2x downsampling MaxPool --
    one VGG-style stage, identical to CNNArchitectureComparison.ipynb's."""
    return [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.MaxPool2d(2),
    ]


class ClassicCNN(nn.Module):
    """Plain, non-grouped Conv2d stack, identical to
    CNNArchitectureComparison.ipynb's ClassicCNN -- reused unmodified here as
    the one architecture every loss variant below trains."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels=28):
        super().__init__()

        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1])
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)

## The loss classes

Five new `AOLoss` subclasses (`ModalMAELoss`, `ModalMSELoss`, `ResidualPhaseMAELoss`, `ResidualPhaseMSELoss`, `NegativeStrehlLoss`), plus `LogResidualVarianceLoss`/`Physics_loss` reused unmodified from `AI4AO/LossFunctions.py`. Every one shares `AOLoss`'s `compute(Ze, z_estimated, pupil, residual_opd, corrected_residual_opd, wfs_frames)` signature (using only the arguments relevant to it), which is what lets `Trainer.train`/`Trainer.evaluate` call any of them uniformly without knowing which one is active.

`ModalMAELoss`/`ModalMSELoss` are the only two that touch `Ze`/`z_estimated` at all -- they supervise the reconstructor's raw modal-coefficient output directly. `ResidualPhaseMAELoss`/`ResidualPhaseMSELoss`/`NegativeStrehlLoss` instead all operate on `corrected_residual_opd` -- the actual post-correction wavefront error -- converted to phase (radians) via the WFS's own sensing wavelength, following `LogResidualVarianceLoss`'s existing convention exactly (`self.pupil`/`self.wavenumber` built once in `__init__`). `Physics_loss` is the odd one out: it never compares against `Ze`, `residual_opd`, or `corrected_residual_opd` as a target at all -- it re-derives a simulated WFS frame from the current correction and penalizes its mismatch against the actual (noisy) observed frame, an indirect, frame-consistency-based training signal rather than a residual-driven one. Keep that qualitative difference in mind when reading its results below: it is not supervising the same kind of error every other candidate here is.

In [ ]:
class ModalMAELoss(AOLoss):
    """Mean absolute error between predicted and true residual modal coefficients."""

    def __init__(self):
        super().__init__()
        self.mae = nn.L1Loss()

    def compute(self, Ze, z_estimated, pupil, residual_opd, corrected_residual_opd, wfs_frames):
        return self.mae(z_estimated, Ze)


class ModalMSELoss(AOLoss):
    """Mean squared error between predicted and true residual modal coefficients."""

    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def compute(self, Ze, z_estimated, pupil, residual_opd, corrected_residual_opd, wfs_frames):
        return self.mse(z_estimated, Ze)


class ResidualPhaseMAELoss(AOLoss):
    """
    Mean absolute post-correction residual phase (radians) over the pupil --
    the phase-domain analogue of ModalMAELoss, supervising the reconstruction
    directly in the sensed phase rather than in modal-coefficient space.
    `corrected_residual_opd` is converted to phase via this WFS's own
    sensing wavelength before taking the mean absolute value, following
    LogResidualVarianceLoss's exact pupil/wavenumber convention.

    Args:
        pupil (Tensor): Binary mask indicating the pupil region.
        wavelength (float): Sensing wavelength (meters) used to convert the
            incoming OPD (meters) to phase (radians).
        device (str): Unused; accepted for interface consistency with other losses.
    """

    def __init__(self, pupil, wavelength, device=None):
        super().__init__()
        self.pupil = torch.clone(pupil).bool()
        wavelength = torch.as_tensor(wavelength, device=self.pupil.device, dtype=torch.float32).reshape(-1)
        self.wavenumber = 2 * torch.pi / wavelength.mean()

    def compute(self, Ze, z_estimated, pupil, residual_opd, corrected_residual_opd, wfs_frames):
        corrected_residual_phase = self.wavenumber * corrected_residual_opd
        return torch.mean(torch.abs(corrected_residual_phase[..., self.pupil]))


class ResidualPhaseMSELoss(AOLoss):
    """
    Mean squared post-correction residual phase (radians) over the pupil --
    the phase-domain analogue of ModalMSELoss. Unlike LogResidualVarianceLoss's
    variance, this does not subtract the per-sample mean phase first, so it
    is the raw second moment of the residual phase rather than its variance.

    Args:
        pupil (Tensor): Binary mask indicating the pupil region.
        wavelength (float): Sensing wavelength (meters) used to convert the
            incoming OPD (meters) to phase (radians).
        device (str): Unused; accepted for interface consistency with other losses.
    """

    def __init__(self, pupil, wavelength, device=None):
        super().__init__()
        self.pupil = torch.clone(pupil).bool()
        wavelength = torch.as_tensor(wavelength, device=self.pupil.device, dtype=torch.float32).reshape(-1)
        self.wavenumber = 2 * torch.pi / wavelength.mean()

    def compute(self, Ze, z_estimated, pupil, residual_opd, corrected_residual_opd, wfs_frames):
        corrected_residual_phase = self.wavenumber * corrected_residual_opd
        return torch.mean(corrected_residual_phase[..., self.pupil] ** 2)


class NegativeStrehlLoss(AOLoss):
    """
    Loss based on the Marechal approximation for Strehl ratio,
    S ~= exp(-var(phase)), applied to the post-correction residual phase
    (radians) over the pupil: same residual_variance LogResidualVarianceLoss
    computes, passed through -exp(.) instead of log(.). Minimizing
    -exp(-var(phase)) directly maximizes this Strehl-ratio approximation;
    unlike log(var(...)), whose magnitude and gradient keep growing without
    bound as the variance shrinks towards zero, this loss saturates near -1
    as the correction improves, mirroring how Strehl itself saturates near 1
    for a well-corrected wavefront.

    Args:
        pupil (Tensor): Binary mask indicating the pupil region.
        wavelength (float): Sensing wavelength (meters) used to convert the
            incoming OPD (meters) to phase (radians).
        device (str): Unused; accepted for interface consistency with other losses.
    """

    def __init__(self, pupil, wavelength, device=None):
        super().__init__()
        self.pupil = torch.clone(pupil).bool()
        wavelength = torch.as_tensor(wavelength, device=self.pupil.device, dtype=torch.float32).reshape(-1)
        self.wavenumber = 2 * torch.pi / wavelength.mean()

    def compute(self, Ze, z_estimated, pupil, residual_opd, corrected_residual_opd, wfs_frames):
        corrected_residual_phase = self.wavenumber * corrected_residual_opd
        residual_variance = torch.var(
            corrected_residual_phase[..., self.pupil], dim=-1, keepdim=True
        )
        return torch.mean(-torch.exp(-residual_variance))

## Instantiating all seven variants

Built once here, sharing `dataset.pupil`/`wfs.wavelength` where relevant -- `LogResidualVarianceLoss` and `Physics_loss` are constructed exactly as the sibling notebooks in this folder already do.

In [ ]:
losses = {
    "ModalMAE": ModalMAELoss(),
    "ModalMSE": ModalMSELoss(),
    "PhaseMAE": ResidualPhaseMAELoss(dataset.pupil, wfs.wavelength),
    "PhaseMSE": ResidualPhaseMSELoss(dataset.pupil, wfs.wavelength),
    "LogVariance": LogResidualVarianceLoss(dataset.pupil, wfs.wavelength),
    "Physics": Physics_loss(wfs=wfs),
    "StrehlApprox": NegativeStrehlLoss(dataset.pupil, wfs.wavelength),
    "Mixed loss": LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs) * 10 + ModalMAELoss() * 10
}

_sanity_model = ClassicCNN(n_channels, Nmodes, Nout)
total_params = sum(p.numel() for p in _sanity_model.parameters() if p.requires_grad)
print(f"ClassicCNN trainable parameters (identical for every loss variant -- capacity never changes here): {total_params:,}")
del _sanity_model

## Training

Following the same procedure as every sibling notebook, repeated once per loss variant: a **fresh `ClassicCNN` instance** (the architecture itself is identical every time, but each variant still needs its own untrained weights) and a fresh `AdamW` optimizer + `Trainer`, all seven sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)` with its own `loss` from the `losses` dict above.

**The same deliberate trade-off every sibling notebook in this folder documents**: `Trainer.train()` calls `self.dataset[0]` fresh at every outer step regardless of which `Trainer` instance is calling it, so training the seven variants one after another on the same shared `dataset` object gives each its own independent i.i.d. atmosphere stream rather than a paired one. Training curves below are directionally informative only; the seeded closed-loop rollout further down is the true apples-to-apples comparison.

Seven full training runs is a real wall-clock cost -- consider a smaller `TrainParams['TrainRunNb']` for an initial correctness pass (confirm every loss converges to something sane) before committing to the full training budget.

In [ ]:
# Closed-loop (BPTT) iterations per optimizer step in the second training stage. Stage 1 uses
# TrainParams['ClosedLoopIterations'] from CNNComparison_params.py (a single pass); this is the
# one training setting that is not in that shared file, so it lives here.
CLOSED_LOOP_ITERATIONS_STAGE2 = 10

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, loss_fn in losses.items():
    print(f"=== Training with {name} loss (stage 1: single pass) ===")

    model = ClassicCNN(n_channels, Nmodes, Nout).to(device=device)
    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss_fn,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by
        # an earlier, differently-sized version of ClassicCNN -- e.g. after
        # editing base_channels above, old checkpoints under PATH are no
        # longer compatible and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal

### Stage 2 -- closed-loop training

The loop above (stage 1) trains every model with `TrainParams['ClosedLoopIterations']` from `CNNComparison_params.py`, i.e. a single pass per optimizer step. The next cell repeats the identical loop with `CLOSED_LOOP_ITERATIONS_STAGE2` closed-loop iterations per optimizer step, continuing from the weights stage 1 just saved (`load_checkpoint` at the top of the loop, and a fresh `AdamW` per model, as in stage 1).

The checkpoint under `PATH` is overwritten with the stage-2 weights, and from here on `trainers`/`loss_trackers`/`loss_trackers_ideal` -- the ones the plotting and evaluation cells below use -- refer to stage 2.

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, loss_fn in losses.items():
    print(f"=== Training with {name} loss (stage 2: closed loop) ===")

    model = ClassicCNN(n_channels, Nmodes, Nout).to(device=device)
    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss_fn,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by
        # an earlier, differently-sized version of ClassicCNN -- e.g. after
        # editing base_channels above, old checkpoints under PATH are no
        # longer compatible and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], CLOSED_LOOP_ITERATIONS_STAGE2)

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal

## Comparing training-loss curves

Unlike the sibling notebooks -- where every variant shares one loss function, so all their curves are the same quantity and can be overlaid on one axis -- each panel below shows one loss variant's own training curve against its own oracle bound (`loss_tracker_ideal`, computed by that same loss instance from the ground-truth residual `Ze`/`ideal_corrected_residual_opd`, per `Trainer.train`'s definition). The two curves within a panel are directly comparable to each other; curves across different panels are not, since they are different physical quantities in different units.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")


fig, axes = plt.subplots(4, 2, figsize=(12, 14))
axes = axes.flatten()

for ax, name in zip(axes, loss_trackers):
    ax.plot(smooth(loss_trackers[name]), label="Training loss")
    ax.plot(smooth(loss_trackers_ideal[name]), label="Ideal (oracle)", color="black", linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Iteration")
    ax.legend()

for ax in axes[len(loss_trackers):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Comparing closed-loop residual wavefront error (nm RMS) and Strehl approximation

The one truly common comparison across all seven: for each variant, reseed immediately before the rollout (the same "reset the seed right before each rollout" trick every sibling notebook uses) so all seven see identical wind/r0/noise draws, then run `trainer.evaluate()` and compute two loss-agnostic metrics from the resulting `residual_opd` trajectory -- steady-state residual wavefront error in nm RMS (`residual_nm_rms`, the same helper/convention every sibling notebook uses), and the actual post-training Marechal-approximation Strehl ratio (`strehl_approx`) -- the literal quantity `NegativeStrehlLoss` is trained to maximize, computed here identically regardless of which loss trained the network. Both exclude the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()` uses internally).

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator
    warm-up period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


def strehl_approx(result, pupil, wavelength, warmup_fraction=0.3):
    """Marechal-approximation Strehl ratio, exp(-var(residual phase)), from an
    EvaluationResult's residual_opd trajectory, converted to phase (radians)
    via the same wavelength/warm-up convention as residual_nm_rms and
    NegativeStrehlLoss above."""
    wavelength = torch.as_tensor(wavelength, dtype=torch.float32).reshape(-1)
    wavenumber = 2 * torch.pi / wavelength.mean()
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state_phase = wavenumber * result.residual_opd[warmup:]
    pupil_values = steady_state_phase[..., pupil.bool()]
    return torch.exp(-torch.var(pupil_values)).item()


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
strehl = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    strehl[name] = strehl_approx(result, dataset.pupil, wfs.wavelength)
    print(f"{name:12s} -- steady-state residual: {nm_rms[name]:6.1f} nm RMS, Strehl approx: {strehl[name]:.3f}")

In [ ]:
names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]
strehl_values = [strehl[n] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(names, rms_values)
axes[0].set_ylabel("Steady-state residual (nm RMS)")
axes[0].set_title("Closed-loop wavefront error")
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, strehl_values)
axes[1].set_ylabel("Strehl approximation (Marechal)")
axes[1].set_title("Closed-loop Strehl approximation")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Visualizing the best loss's closed loop

A closer look at whichever loss came out on top above (lowest nm RMS) -- the same rollout-animation pattern every sibling notebook in this folder uses.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best loss: {best_name} ({nm_rms[best_name]:.1f} nm RMS, Strehl approx {strehl[best_name]:.3f})")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number=9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())